In [1]:
using IJulia
println("Julia kernel is active!")
println("Julia version: ", VERSION)

Julia kernel is active!
Julia version: 1.10.4


In [2]:
using SparseArrays
using LinearAlgebra
using Dates
using DelimitedFiles
using Random

In [3]:
"""
    random_row_submatrix(M::AbstractMatrix, k::Int; seed::Union{Int,Nothing}=nothing)

Selects k random rows from incidence matrix M and returns the submatrix.
If seed is provided, randomness is reproducible.
"""
function random_row_submatrix(M::AbstractMatrix, k::Int; seed::Union{Int,Nothing}=nothing)
    n_rows = size(M, 1)
    if k > n_rows
        throw(ArgumentError("k ($k) cannot exceed number of rows ($n_rows)"))
    end
    if seed !== nothing
        Random.seed!(seed)
    end
    selected_rows = shuffle(1:n_rows)[1:k]
    return M[selected_rows, :]
end

random_row_submatrix

In [ ]:
"""
    incidence_matrix(hyperedges::Vector{<:AbstractVector}; sparse_output::Bool=false)

Given a hypergraph as a vector of vectors (each inner vector is a hyperedge listing its vertices),
returns the incidence matrix as an AbstractMatrix.
Each row is a hyperedge, each column is a vertex.
Set `sparse_output=true` to return a sparse matrix.
"""
function incidence_matrix(hyperedges::Vector{<:AbstractVector}; sparse_output::Bool=false)
    vertices = sort(unique(vcat(hyperedges...)))
    num_edges = length(hyperedges)
    num_vertices = length(vertices)
    vertex_to_col = Dict(v => i for (i, v) in enumerate(vertices))

    if sparse_output
        row_indices = Int[]
        col_indices = Int[]
        values = Int[]
        for (i, edge) in enumerate(hyperedges)
            for v in edge
                push!(row_indices, i)
                push!(col_indices, vertex_to_col[v])
                push!(values, 1)
            end
        end
        return sparse(row_indices, col_indices, values, num_edges, num_vertices)
    else
        M = zeros(Int, num_edges, num_vertices)
        for (i, edge) in enumerate(hyperedges)
            for v in edge
                M[i, vertex_to_col[v]] = 1
            end
        end
        return M
    end
end

# Example usage:
# hyperedges = [[1,2,3], [1,2], [3,4,5,6]]
# M_dense = incidence_matrix(hyperedges)
# M_sparse = incidence_matrix(hyperedges; sparse_output=true)

In [ ]:



function Frobenius3Product(A::AbstractArray, B::AbstractArray, C::AbstractArray)
    # Check that all tensors have the same shape
    if !(size(A) == size(B) == size(C))
        throw(ArgumentError("A, B, and C must have the same shape"))
    end

    shape = size(A)
    l = length(shape)
    n = shape[1]
    result = similar(A)

    for idx in CartesianIndices(shape)
        inds = Tuple(idx)
        s = zero(eltype(A))
        for j in 1:n
            indsA = ntuple(m -> m == 1 ? j : inds[m], l)
            indsB = ntuple(m -> m == 2 ? j : inds[m], l)
            indsC = ntuple(m -> m == 3 ? j : inds[m], l)
            s += A[indsA...] * B[indsB...] * C[indsC...]
        end
        result[idx] = s
    end
    return result
end


function barycentric_tensor(A::AbstractArray)
    shape = size(A)
    l = length(shape)
    n = shape[1]
    if any(x -> x != n, shape)
        throw(ArgumentError("Input tensor must be symmetric (all dimensions equal), got shape $shape"))
    end
    B = similar(A)

    for idx in CartesianIndices(shape)
        inds = Tuple(idx)
        s = zero(eltype(A))
        for j in 1:n
            prod = one(eltype(A))
            for k in 1:l
                inds_k = ntuple(m -> m == k ? j : inds[m], l)
                prod *= A[inds_k...]
            end
            s += prod
        end
        B[idx] = s
    end
    return B
end


"""
    barycentric_tensor(M::AbstractMatrix)

Given an incidence matrix M (no all-zero rows/columns, constant row sum),
returns a barycentric matrix B of the same size.
"""
function barycentric_Incidence_tensor(M::AbstractMatrix)
    n_rows, n_cols = size(M)
    m = sum(M[1, :])  # Assumes all rows have the same sum
    B = zeros(eltype(M), n_rows, n_cols)
    for i in 1:n_rows
        for j in 1:n_cols
            s = zero(eltype(M))
            # For each possible barycentric product, sum over all row indices
            for k in 1:n_rows
                prod = one(eltype(M))
                for l in 1:m
                    prod *= M[k, j]
                end
                s += prod
            end
            B[i, j] = s
        end
    end
    return B
end




barycentric_Incidence_tensor

In [43]:


function iterated_barycentric_tensors(A; iters=nothing, max_minutes=nothing)
    tensors = [A]
    println("Computing iterated barycentric tensors:")
    println("  T_0 (original tensor):")
    # println(A)
    start_time = now()
    i = 1
    while true
        # Check time limit
        if max_minutes !== nothing
            elapsed = (now() - start_time).value / (60 * 1e9) # minutes
            if elapsed > max_minutes
                println("Time limit reached after $i iterations ($elapsed minutes).")
                return tensors, false, i
            end
        end
        # Check iteration limit
        if iters !== nothing && i > iters
            println("Max iterations ($iters) reached.")
            return tensors, false, iters
        end
        # Print iteration info if both limits are unbounded
        if iters === nothing && max_minutes === nothing
            println("  Iteration $i (no bounds):")
        else
            println("  Computing tensor T_", i, " ...")
        end
        next_tensor = barycentric_tensor(tensors[end])
        push!(tensors, next_tensor)
        println("  T_", i, ":")
        # println(next_tensor)
        println("    Done with T_", i)
        try
            coeffs = find_polynomial_relation(tensors)
            println("Polynomial relation coefficients found at iteration $i: ", coeffs)
            println("Terminating early due to successful relation.")
            return tensors, true, i
        catch e
            if isa(e, ErrorException)
                # No relation found yet, continue
            else
                rethrow(e)
            end
        end
        i += 1
    end
end


function find_polynomial_relation(tensors)
    n = length(tensors)
    if n < 2
        error("Need at least two tensors to find a relation")
    end
    mats = [vec(T) for T in tensors]
    M = hcat(mats[1:end-1]...)
    b = mats[end]
    # Solve least squares M * u ≈ b
    u = M \ b
    # Check if the solution is nontrivial (not all zeros, and residual is small)
    if all(iszero, u) || norm(M * u - b) > 1e-8 * norm(b)
        error("No nontrivial polynomial relation found (increase number of iterations)")
    end
    coeffs = vcat(-u, 1)
    return coeffs
end



find_polynomial_relation (generic function with 1 method)

In [20]:
"""
    convert_and_load_hyperedges(nverts_file::String, simplices_file::String, output_file::String)

Reads the contact-high-school dataset files, writes hyperedges to a .txt file in the format [[1,2,3], ...], and loads them as a Julia variable.

Returns: hyperedges::Vector{Vector{Int}}
"""
function convert_and_load_hyperedges(nverts_file::String, simplices_file::String, output_file::String)
    # Step 1: Parse files and extract hyperedges
    nverts = vec(readdlm(nverts_file, Int))
    all_nodes = vec(readdlm(simplices_file, Int))
    hyperedges = Vector{Vector{Int}}()
    node_idx = 1
    for nvert in nverts
        simplex_nodes = all_nodes[node_idx:(node_idx + nvert - 1)]
        push!(hyperedges, simplex_nodes)
        node_idx += nvert
    end

    # Step 2: Write hyperedges to .txt file
    open(output_file, "w") do io
        print(io, "[")
        for (i, edge) in enumerate(hyperedges)
            print(io, "[", join(edge, ","), "]")
            if i < length(hyperedges)
                print(io, ", ")
            end
        end
        println(io, "]")
    end
    println("✅ Hyperedges written to $output_file")

    # Step 3: Read .txt file and load as Julia variable
    txt = read(output_file, String)
    A = eval(Meta.parse(txt))
    println("✅ Loaded hyperedges into variable A")
    println("First 5 hyperedges: ", A[1:5])
    return A
end

# Example usage:
# A = convert_and_load_hyperedges(
#     "contact-high-school/contact-high-school-nverts.txt",
#     "contact-high-school/contact-high-school-simplices.txt",
#     "contact-high-school-hyperedges.txt"
# )

convert_and_load_hyperedges

In [21]:


function createTensorFromIncidence3(M::AbstractMatrix{T}) where T
    # Check that M is a 01 matrix
    if !all(x -> x == 0 || x == 1, M)
        throw(ArgumentError("Matrix M must be a 0-1 matrix"))
    end
    
    d = size(M, 2)  # number of columns
    e = size(M, 1)  # number of rows
    
    # Check that each row has exactly 3 ones
    for row in 1:e
        if sum(M[row, :]) != 3
            throw(ArgumentError("Each row must have exactly 3 ones"))
        end
    end
    
    # Create d x d x d tensor initialized to zeros
    t = zeros(Float64, d, d, d)
    
    # For each row in M, find the three nonzero columns and set t[i,j,k] = 1
    for row in 1:e
        nonzero_cols = findall(x -> x == 1, M[row, :])
        if length(nonzero_cols) == 3
            i, j, k = nonzero_cols
            t[i, j, k] = 1
            t[i, k, j] = 1
            t[j, i, k] = 1
            t[j, k, i] = 1
            t[k, i, j] = 1
            t[k, j, i] = 1
        end
    end
    
    return t
end





function createTensorFromIncidence(M::AbstractMatrix{T}, m::Integer; field::Type=Float64) where T
    # Check that M is a 0-1 matrix
    if !all(x -> x == 0 || x == 1, M)
        throw(ArgumentError("Matrix M must be a 0-1 matrix"))
    end
    if m < 1
        throw(ArgumentError("m must be >= 1"))
    end

    d = size(M, 2)  # number of columns i.e. the number of vertices
    e = size(M, 1)  # number of rows i.e. the number of (hyper) edges
    if m > d #m is looking for a sub m-uniform hypergraph, you need at least m vertices to make 
        #a hyperedge with exactly m vertices.
        throw(ArgumentError("m cannot exceed number of columns")) 
    end

    # Create an m-way tensor of size d x d x ... x d with specified element type
    dims = ntuple(_ -> d, m)
    t = zeros(field, dims...)

    # recursive helper to set all permutations of indices in tensor to one(field)
    function set_permutations!(Tarr, inds::Vector{Int}, i::Int)
        if i == length(inds)
            Tarr[Tuple(inds)...] = one(field)
            return
        end
        for j in i:length(inds)
            inds[i], inds[j] = inds[j], inds[i]
            set_permutations!(Tarr, inds, i + 1)
            inds[i], inds[j] = inds[j], inds[i]
        end
    end

    # Process rows that have exactly m ones
    for row in 1:e
        cols = findall(x -> x == 1, M[row, :])
        if length(cols) == m
            set_permutations!(t, collect(cols), 1)
        end
    end

    return t
end




function simplicial_3_complex(M::AbstractMatrix{Int})
    # Find all unique vertices (columns)
    vertices = collect(1:size(M, 2))
    three_edges = Vector{Vector{Int}}()
    for row in 1:size(M, 1)
        cols = findall(x -> x == 1, M[row, :])
        if length(cols) == 3
            push!(three_edges, cols)
        elseif length(cols) > 3
            # Generate all 3-element combinations
            for i in 1:length(cols)-2
                for j in i+1:length(cols)-1
                    for k in j+1:length(cols)
                        push!(three_edges, [cols[i], cols[j], cols[k]])
                    end
                end
            end
        end
    end
    # Remove duplicate hyperedges (sorted for canonical form)
    unique_edges = unique(sort.(three_edges))
    # Build new incidence matrix
    num_edges = length(unique_edges)
    num_vertices = length(vertices)
    new_M = zeros(Int, num_edges, num_vertices)
    for (i, edge) in enumerate(unique_edges)
        for v in edge
            new_M[i, v] = 1
        end
    end
    return new_M
end


function random_row_submatrix(M::AbstractMatrix, k::Int; seed::Union{Int,Nothing}=nothing)
    n_rows = size(M, 1)
    if k > n_rows
        throw(ArgumentError("k ($k) cannot exceed number of rows ($n_rows)"))
    end
    if seed !== nothing
        Random.seed!(seed)
    end
    selected_rows = shuffle(1:n_rows)[1:k]
    subM = M[selected_rows, :]
    # Remove columns with all zeros
    nonzero_cols = findall(col -> any(subM[:, col] .!= 0), 1:size(subM, 2))
    return subM[:, nonzero_cols]
end

# Example usage:
# M = incidence_matrix([[1,2,3,4], [2,3,4], [1,2,3]])
# M3 = simplicial_3_complex(M)

random_row_submatrix (generic function with 1 method)

In [69]:
# incidence=[1 1 1 0 0 0; 1 1 0 1 0 0; 0 1 1 1 0 0; 0 0 1 1 1 0; 1 0 0 1 1 0; 1 0 1 0 1 0; 1 0 0 0 1 1; 0 0 1 0 1 1; 1 0 1 0 0 1]

# incidence=[1 1 1 0 0 0; 1 1 0 1 0 0; 0 1 1 1 0 0; 1 0 1 1 0 0 ;0 0 1 1 1 0; 1 0 0 1 1 0; 1 0 1 0 1 0; 1 0 0 0 1 1; 0 0 1 0 1 1; 1 0 1 0 0 1] # up to six vertices

# incidence= [1 1 1 0; 1 1 0 1; 1 0 1 1; 0 1 1 1] #up to four vertices

# incidence = [1 1 1 0 0; 1 1 0 1 0; 1 0 1 1 0; 0 1 1 1 0; 1 0 1 0 1; 1 0 0 1 1; 0 0 1 1 1] #up to five vertices

# incidence=[1 1 1 0 0 0 0; 1 1 0 1 0 0 0; 0 1 1 1 0 0 0; 0 0 1 1 1 0 0; 1 0 0 1 1 0 0; 1 0 1 0 1 0 0; 1 0 0 0 1 1 0; 0 0 1 0 1 1 0; 1 0 1 0 0 1 0;1 0 0 0 0 1 1; 0 0 1 0 0 1 1; 1 0 1 0 0 0 1]# up to seven vertices

# incidence=[1 1 1 0 0 0 0 0; 1 1 0 1 0 0 0 0; 0 1 1 1 0 0 0 0; 0 0 1 1 1 0 0 0; 1 0 0 1 1 0 0 0; 1 0 1 0 1 0 0 0; 1 0 0 0 1 1 0 0; 0 0 1 0 1 1 0 0; 1 0 1 0 0 1 0 0;1 0 0 0 0 1 1 0 ; 0 0 1 0 0 1 1 0; 1 0 1 0 0 0 1 0 ;1 0 0 0 0 0 1 1; 0 0 1 0 0 0 1 1; 1 0 1 0 0 0 0 1] # up to eight


"""
    barycentric_subdivision_incidence(n::Int)

Generate the incidence matrix for the n-th barycentric subdivision of a tetrahedron.
Each row corresponds to a 3-element subset (triangle), each column to a vertex.
"""

function barycentric_subdivision_incidence(n::Int)
    if n < 1
        error("n must be at least 1")
    end

    # Base case: n = 1
    if n == 1
        # Vertices: 1,2,3,4
        # Triangles: {1,2,3}, {1,2,4}, {1,3,4}, {2,3,4}
        return [
            1 1 1 0;
            1 1 0 1;
            1 0 1 1;
            0 1 1 1
        ]
    end

    prev = barycentric_subdivision_incidence(n-1)
    num_vertices = size(prev, 2)
    new_vertex = num_vertices + 1

    # Add three new triangles (rows)
    # {1,3,new_vertex}
    row1 = zeros(Int, num_vertices + 1)
    row1[1] = 1
    row1[3] = 1
    row1[new_vertex] = 1

    # {1,num_vertices,new_vertex}
    row2 = zeros(Int, num_vertices + 1)
    row2[1] = 1
    row2[num_vertices] = 1
    row2[new_vertex] = 1

    # {3,num_vertices,new_vertex}
    row3 = zeros(Int, num_vertices + 1)
    row3[3] = 1
    row3[num_vertices] = 1
    row3[new_vertex] = 1

    # Expand previous matrix to new number of columns
    prev_expanded = hcat(prev, zeros(Int, size(prev,1), 1))

    # Stack all together
    return vcat(prev_expanded, row1', row2', row3')
end
# Example usage:
# barycentric_subdivision_incidence(1)
# barycentric_subdivision_incidence(2)

using Random

"""
    random_barycentric_subdivision_incidence(n::Int; rng=Random.GLOBAL_RNG)

Generate the incidence matrix for the n-th barycentric subdivision of a tetrahedron,
but at each step, the three new triangles are formed using random (distinct) vertices
from the current set, together with the new vertex.
"""
function random_barycentric_subdivision_incidence(n::Int; rng=Random.GLOBAL_RNG)
    if n < 1
        error("n must be at least 1")
    end

    # Base case: n = 1
    if n == 1
        return [
            1 1 1 0;
            1 1 0 1;
            1 0 1 1;
            0 1 1 1
        ]
    end

    prev = random_barycentric_subdivision_incidence(n-1; rng=rng)
    num_vertices = size(prev, 2)
    new_vertex = num_vertices + 1

    # Pick three distinct random vertices for each new triangle
    verts = collect(1:num_vertices)
    randoms = randperm(rng, num_vertices)[1:3]
    r1, r2, r3 = randoms

    # {r1, r2, new_vertex}
    row1 = zeros(Int, num_vertices + 1)
    row1[r1] = 1
    row1[r2] = 1
    row1[new_vertex] = 1

    # {r1, r3, new_vertex}
    row2 = zeros(Int, num_vertices + 1)
    row2[r1] = 1
    row2[r3] = 1
    row2[new_vertex] = 1

    # {r3, r2, new_vertex}
    row3 = zeros(Int, num_vertices + 1)
    row3[r3] = 1
    row3[r2] = 1
    row3[new_vertex] = 1

    prev_expanded = hcat(prev, zeros(Int, size(prev,1), 1))
    return vcat(prev_expanded, row1', row2', row3')
end

# Example usage:
# rng = MersenneTwister(123)
# random_barycentric_subdivision_incidence(2; rng=rng)

# incidence = barycentric_subdivision_incidence(2)
# incidence = random_barycentric_subdivision_incidence(3)
# print(incidence)
incidence = [1 1 0 1; 0 1 1 1; 1 0 1 1]
# incidence = [1 1 0 1 0; 1 0 0 1 1; 0 1 1 1 0; 0 0 1 1 1; 1 0 1 0 1]
# incidence = [1 1 0 1 0; 1 0 0 1 1; 0 1 1 1 0; 0 0 1 1 1; 1 0 1 0 1]


tensor=createTensorFromIncidence3(incidence)
test_bary=barycentric_tensor(tensor)
test_bary2=barycentric_tensor(test_bary)
# # test_bary
# tensors=iterated_barycentric_tensors(tensor)


4×4×4 Array{Float64, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0

[:, :, 2] =
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0

[:, :, 3] =
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0

[:, :, 4] =
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0

In [10]:
tensors[1]

3-element Vector{Array{Float64, 3}}:
 [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 1.0 0.0; … ; 0.0 1.0 … 0.0 1.0; 0.0 0.0 … 1.0 0.0;;; 0.0 0.0 … 1.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 1.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;; 0.0 1.0 … 0.0 1.0; 1.0 0.0 … 1.0 0.0; … ; 0.0 1.0 … 0.0 1.0; 1.0 0.0 … 1.0 0.0;;; 0.0 1.0 … 0.0 1.0; 1.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 1.0 0.0 … 0.0 0.0;;; 0.0 0.0 … 1.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 1.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]
 [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;; 0.0 0.0 … 2.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 2.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]
 [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0

In [ ]:
edge_to_incidence_matrix = incidence_matrix(A)
high_school_3_hyperedges = simplicial_3_complex(edge_to_incidence_matrix)
high_school_full_3_tensor = createTensorFromIncidence3(high_school_3_hyperedges)
iterated_barycentric_tensors(high_school_full_3_tensor,iters=20,max_minutes=10)

In [ ]:
k=20 #Randomly choose 20 hyperedges
random_high_school= random_row_submatrix(high_school_3_hyperedges, k)
high_school_full_3_tensor = createTensorFromIncidence3(random_high_school)
iterated_barycentric_tensors(high_school_full_3_tensor,iters=20,max_minutes=10)